In [1]:
%load_ext autoreload
%autoreload 2

# New model

In [4]:
import pandas as pd
import yaml
from prophet import Prophet
from prophet.utils import validate_prophet_inputs

In [5]:
pretrained_checkpoint_path = "/ictstr01/groups/ml01/projects/super_rad_project/pretrained_prophet/everything/iv_0_iv_out_multitest_300cl_1219iv_512model_8layers_Falsesimpler_Truemask_0.0001lr_Falseexplicitphenotype_10000warmup_150001max_iters_Falseunbalanced_0.01wd_4096bs_Falseft/iv_0_seed_110/epoch=25-step=37648.ckpt"
model = Prophet(
    iv_emb_path=[
        "/lustre/groups/ml01/projects/super_rad_project/intervention_embeddings/global_iv_scaledv3.csv"
    ],
    cl_emb_path=[
        "/lustre/groups/ml01/projects/super_rad_project/cell_line_embeddings/cell_line_embedding_full_ccle_300_scaled.csv"
    ],
    ph_emb_path=None,
    model_pth=pretrained_checkpoint_path,
)

Learning rate set to 1e-05


In [6]:
iv_list = [
    "C(Cc1c[nH]c2ccccc12)Nc1cccc(Nc2ccncc2)c1",
    "C(Cc1c[nH]c2ccccc12)Nc1cccc(Nc2ccncc2)c1",
    "C(Cc1c[nH]c2ccccc12)Nc1cccc(Nc2ccncc2)c1",
    "C(Cc1c[nH]c2ccccc12)Nc1cccc(Nc2ccncc2)c1",
]
cl_list = ["A375", "UACC62", "K562", "JURKAT"]
ph_list = ["GDSC"]

In [7]:
source_df = pd.DataFrame(
    {
        "iv1": iv_list,
        "cell_line": cl_list,
        "phenotype": ph_list[0],
        "_": 0.0,  # dummy for prediction
    }
)
source_df["iv2"] = "negative_gene"
source_df

,iv1,cell_line,phenotype,_,iv2
0,C(Cc1c[nH]c2ccccc12)Nc1cccc(Nc2ccncc2)c1,A375,GDSC,0.0,negative_gene
1,C(Cc1c[nH]c2ccccc12)Nc1cccc(Nc2ccncc2)c1,UACC62,GDSC,0.0,negative_gene
2,C(Cc1c[nH]c2ccccc12)Nc1cccc(Nc2ccncc2)c1,K562,GDSC,0.0,negative_gene
3,C(Cc1c[nH]c2ccccc12)Nc1cccc(Nc2ccncc2)c1,JURKAT,GDSC,0.0,negative_gene


In [8]:
validation_results = validate_prophet_inputs(
    df=source_df,
    iv_emb_path=model.iv_emb_path,
    cl_emb_path=model.cl_emb_path,
    ph_emb_path=model.ph_emb_path,
    iv_col=["iv1", "iv2"],
    cl_col="cell_line",
    ph_col="phenotype",
    readout_col="response",
    mode="predict",
)

/ictstr01/home/icb/alejandro.tejada/prophet/prophet/utils/validation.py:50: UserWarning: Unexpected columns found: ['_']. These will be ignored during processing.
  warnings.warn(


In [9]:
source_df = validation_results["processed_inputs"]["df"]

In [10]:
source_df

,iv1,cell_line,phenotype,_,iv2
0,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,A375,gdsc,0.0,negative_gene
1,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,UACC62,gdsc,0.0,negative_gene
2,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,K562,gdsc,0.0,negative_gene
3,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,JURKAT,gdsc,0.0,negative_gene


In [12]:
# predict with lists of treatments and cell lines
df = model.predict(
    source_df,
    save=False,
)
df

/ictstr01/groups/ml01/workspace/alejandro.tejada/micromamba/envs/prophet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/ictstr01/groups/ml01/workspace/alejandro.tejada/micromamba/envs/prophet/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3.11 /ictstr01/groups/ml01/workspace/alejandro.tejada ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org

Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]


,iv1,cell_line,phenotype,iv2,pred
0,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,A375,gdsc,negative_gene,0.086387
1,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,UACC62,gdsc,negative_gene,0.126224
2,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,K562,gdsc,negative_gene,0.111408
3,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,JURKAT,gdsc,negative_gene,0.350501


# Old implementation

In [86]:
import os

In [87]:
os.chdir(
    "/lustre/groups/ml01/workspace/alejandro.tejada/safe_super_rad_project/super_rad_project"
)

In [88]:
import torch
import pandas as pd
from Transformer import TransformerPredictor
import pytorch_lightning as pl

In [94]:
from prophet.data import process_priors

In [95]:
iv, cl, phe = process_priors(
    [
        "/lustre/groups/ml01/projects/super_rad_project/intervention_embeddings/global_iv_scaledv3.csv"
    ],
    [
        "/lustre/groups/ml01/projects/super_rad_project/cell_line_embeddings/cell_line_embedding_full_ccle_300_scaled.csv"
    ],
    None,
)

In [89]:
model = TransformerPredictor.load_from_checkpoint(pretrained_checkpoint_path)
print("✅ Model loaded successfully!")
print(f"Model device: {model.device}")

Gene net:  Sequential(
  (0): Linear(in_features=1219, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Cell line net:  Sequential(
  (0): Linear(in_features=300, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Regressor:  Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): GELU(approximate='none')
  (5): Linear(in_features=512, out_features=1, bias=True)
)
✅ Model loaded successfully!
Model device: cuda:0


In [97]:
source_df

,iv1,cell_line,phenotype,_,iv2
0,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,A375,gdsc,0.0,negative_gene
1,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,UACC62,gdsc,0.0,negative_gene
2,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,K562,gdsc,0.0,negative_gene
3,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,JURKAT,gdsc,0.0,negative_gene


In [98]:
source_df["value"] = 1.0

In [100]:
source_df.loc[0, "value"] = 0.0

In [101]:
source_df

,iv1,cell_line,phenotype,_,iv2,value
0,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,A375,gdsc,0.0,negative_gene,0.0
1,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,UACC62,gdsc,0.0,negative_gene,1.0
2,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,K562,gdsc,0.0,negative_gene,1.0
3,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,JURKAT,gdsc,0.0,negative_gene,1.0


In [104]:
from dataloader import dataloader_regression_sources
from config import set_config
import yaml

data = dataloader_regression_sources(
    gene_embedding=iv,
    cell_lines_embedding=cl,
    phenotype_embedding=None,
    data_label=source_df,
    label_name="value",
    index=(list(source_df.index), [], list(source_df.index), []),
    batch_size=32,
    unbalanced=False,
    pert_len=2,
    phenotypes=model.hparams.phenotypes,
)

test_dataloader = data[2]  # test dataloader

# Run inference
trainer = pl.Trainer(accelerator="gpu" if torch.cuda.is_available() else "cpu")
predictions = trainer.predict(model, test_dataloader)

# Extract results
all_predictions = torch.cat([p[0] for p in predictions], dim=0)
print(f"Predictions shape: {all_predictions.shape}")
print(f"Predictions: {all_predictions}")

/ictstr01/groups/ml01/workspace/alejandro.tejada/micromamba/envs/prophet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/ictstr01/groups/ml01/workspace/alejandro.tejada/micromamba/envs/prophet/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3.11 /ictstr01/groups/ml01/workspace/alejandro.tejada ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org

Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 19.36it/s]
Predictions shape: torch.Size([4, 1])
Predictions: tensor([[0.0864],
        [0.1262],
        [0.1114],
        [0.3505]])


In [105]:
for batch in test_dataloader:
    print(batch)

{'phenotype': tensor([54, 54, 54, 54]), 'cell_line': tensor([[-0.2032, -0.9007,  0.0495,  ..., -0.7714,  0.3555,  0.9185],
        [-0.3126, -1.0483,  0.3018,  ..., -0.5802, -0.7692, -1.1889],
        [ 1.1609, -0.1581, -0.3007,  ..., -0.2113, -1.2081, -0.1626],
        [ 1.6679, -0.6565, -1.0046,  ...,  0.9192,  0.0326,  1.9326]],
       dtype=torch.float64), 'label': tensor([0., 1., 1., 1.], dtype=torch.float64), 'attn_mask': tensor([[False, False,  True, False, False],
        [False, False,  True, False, False],
        [False, False,  True, False, False],
        [False, False,  True, False, False]]), 'idx': tensor([0, 1, 2, 3]), 'pert_type': tensor([[1, 0],
        [1, 0],
        [1, 0],
        [1, 0]]), 'cell_line_name': ['A375', 'UACC62', 'K562', 'JURKAT'], 'phenotype_name': ['gdsc', 'gdsc', 'gdsc', 'gdsc'], 'iv_names': {0: ['c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1', 'c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1', 'c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1', 'c(cc1c[nh]c2ccccc12)

In [113]:
for batch in test:
    print(batch)

/ictstr01/groups/ml01/workspace/alejandro.tejada/micromamba/envs/prophet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


{'phenotype': tensor([999, 999, 999, 999]), 'cell_line': tensor([[-0.2032, -0.9007,  0.0495,  ..., -0.7714,  0.3555,  0.9185],
        [-0.3126, -1.0483,  0.3018,  ..., -0.5802, -0.7692, -1.1889],
        [ 1.1609, -0.1581, -0.3007,  ..., -0.2113, -1.2081, -0.1626],
        [ 1.6679, -0.6565, -1.0046,  ...,  0.9192,  0.0326,  1.9326]],
       dtype=torch.float64), 'label': tensor([0., 0., 0., 0.]), 'attn_mask': tensor([[False, False,  True, False, False],
        [False, False,  True, False, False],
        [False, False,  True, False, False],
        [False, False,  True, False, False]]), 'idx': tensor([0, 1, 2, 3]), 'pert_type': tensor([[-1.0099,  0.0000],
        [-1.0099,  0.0000],
        [-1.0099,  0.0000],
        [-1.0099,  0.0000]], dtype=torch.float64), 'iv1': tensor([[-1.0099,  0.2204, -0.3278,  ..., -0.2101, -0.1591, -0.1800],
        [-1.0099,  0.2204, -0.3278,  ..., -0.2101, -0.1591, -0.1800],
        [-1.0099,  0.2204, -0.3278,  ..., -0.2101, -0.1591, -0.1800],
        [